# ForestGuard

## 04 — Modelos de regresión

Entrenamiento y comparación de modelos de regresión con el dataset procesado.

> **Nota:** Este notebook corresponde a una etapa preliminar y complementaria de ForestGuard.
>
> Su objetivo es estimar el número de hectáreas afectadas (`hadecobert`) en incendios ya registrados. Esta tarea no representa directamente la probabilidad de ocurrencia de un incendio.
>
> El modelo principal de riesgo se desarrolla en `06_modelo_riesgo_incendio.ipynb`.

## 1. Carga del dataset procesado

Se utiliza el archivo generado en la etapa de Big Data.

In [ ]:
from pathlib import Path
from google.colab import files
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

ROOT = Path("/content")
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

archivo_processed = PROCESSED / "incendios_forestales_2022_2025_limpio.csv"

if not archivo_processed.exists():
    print("Sube el archivo generado por el módulo 02:")
    print("- incendios_forestales_2022_2025_limpio.csv")

    subidos = files.upload()

    for nombre, contenido in subidos.items():
        with open(PROCESSED / nombre, "wb") as f:
            f.write(contenido)

if not archivo_processed.exists():
    raise FileNotFoundError(
        "No se encontró el dataset procesado. "
        "Ejecuta el módulo 02 o sube el CSV generado."
    )

df = pd.read_csv(archivo_processed)
print("Dataset cargado:", df.shape)

## 2. Selección de variables

Se incorpora la variable `año` junto con variables temporales, geográficas y de impacto disponibles en el dataset integrado.

Las variables utilizadas describen incendios ya registrados. Por este motivo, este modelo se interpreta como una aproximación para estudiar la magnitud de la afectación y no como un sistema definitivo de predicción de riesgo.

In [ ]:
objetivo = "hadecobert"

variables_numericas = [
    "año",
    "mes",
    "dia",
    "latitud",
    "longitud",
    "personas_afectadas",
    "familias_afectadas",
    "animales_afectados",
    "animales_muertos",
]

variables_categoricas = [
    "provincia",
    "canton",
    "region",
]

variables_numericas = [
    c for c in variables_numericas
    if c in df.columns
]

variables_categoricas = [
    c for c in variables_categoricas
    if c in df.columns
]

variables = variables_numericas + variables_categoricas

datos_modelo = df[variables + [objetivo]].copy()
datos_modelo[objetivo] = pd.to_numeric(
    datos_modelo[objetivo],
    errors="coerce"
)

datos_modelo = datos_modelo.dropna(subset=[objetivo]).copy()

print("Registros disponibles para regresión:", datos_modelo.shape[0])
print("Variables utilizadas:", variables)

## 3. División de entrenamiento y prueba

Se utiliza el 80 % de los registros para entrenamiento y el 20 % para evaluación. Se fija `random_state=10` para facilitar la reproducibilidad.

In [ ]:
X = datos_modelo[variables]
y = datos_modelo[objetivo]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=10
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

## 4. Preprocesamiento

Las variables numéricas se completan con la mediana y se estandarizan.

Las variables categóricas se completan con la categoría más frecuente y posteriormente se transforman mediante One-Hot Encoding.

In [ ]:
preprocesamiento_numerico = Pipeline([
    ("imputacion", SimpleImputer(strategy="median")),
    ("escalado", StandardScaler())
])

preprocesamiento_categorico = Pipeline([
    ("imputacion", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocesador = ColumnTransformer([
    (
        "numericas",
        preprocesamiento_numerico,
        variables_numericas
    ),
    (
        "categoricas",
        preprocesamiento_categorico,
        variables_categoricas
    )
])

## 5. Modelos implementados

Se comparan dos algoritmos:

1. **Regresión Lineal:** sirve como modelo sencillo de referencia.
2. **Random Forest Regressor:** permite capturar relaciones no lineales entre las variables.

También se utiliza `DummyRegressor` como baseline. Su función es comprobar si los modelos realmente mejoran frente a una predicción muy básica basada en la media.

In [ ]:
modelos = {
    "Regresión Lineal": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=10,
        n_jobs=-1
    ),
    "Baseline (media)": DummyRegressor(
        strategy="mean"
    )
}

## 6. Entrenamiento y evaluación

Se utilizan las métricas MSE, RMSE, MAE y R².

- **MSE:** penaliza con mayor intensidad los errores grandes.
- **RMSE:** expresa el error en la misma unidad que las hectáreas.
- **MAE:** representa el error absoluto promedio.
- **R²:** indica qué proporción de la variación es explicada por el modelo.

In [ ]:
resultados = []
modelos_entrenados = {}

for nombre, modelo in modelos.items():

    pipeline = Pipeline([
        ("preprocesamiento", preprocesador),
        ("modelo", modelo)
    ])

    pipeline.fit(X_train, y_train)

    predicciones = pipeline.predict(X_test)

    mse = mean_squared_error(y_test, predicciones)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, predicciones)
    r2 = r2_score(y_test, predicciones)

    resultados.append({
        "Modelo": nombre,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

    modelos_entrenados[nombre] = pipeline

resultados_regresion = pd.DataFrame(resultados)
resultados_regresion = resultados_regresion.sort_values("RMSE")

display(resultados_regresion.round(4))

## 7. Comparación gráfica

In [ ]:
grafico = resultados_regresion.set_index("Modelo")[["RMSE", "MAE"]]

grafico.plot(
    kind="bar",
    figsize=(9, 5)
)

plt.title("Comparación de modelos de regresión")
plt.xlabel("Modelo")
plt.ylabel("Error")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 8. Selección preliminar del mejor modelo

El modelo con menor RMSE entre los modelos reales se considera la mejor alternativa preliminar.

El baseline no se selecciona como modelo final; únicamente sirve como referencia.

In [ ]:
modelos_reales = resultados_regresion[
    ~resultados_regresion["Modelo"].str.contains("Baseline")
].copy()

mejor_modelo_regresion = (
    modelos_reales
    .sort_values("RMSE")
    .iloc[0]
)

print("Mejor modelo de regresión:")
print(mejor_modelo_regresion)

## 9. Predicciones frente a valores reales

In [ ]:
nombre_mejor = mejor_modelo_regresion["Modelo"]
modelo_mejor = modelos_entrenados[nombre_mejor]

pred_mejor = modelo_mejor.predict(X_test)

plt.figure(figsize=(7, 6))
plt.scatter(y_test, pred_mejor, alpha=0.7)

minimo = min(y_test.min(), pred_mejor.min())
maximo = max(y_test.max(), pred_mejor.max())

plt.plot(
    [minimo, maximo],
    [minimo, maximo],
    linestyle="--"
)

plt.title(f"Valores reales vs. predichos — {nombre_mejor}")
plt.xlabel("Hectáreas reales")
plt.ylabel("Hectáreas predichas")
plt.tight_layout()
plt.show()

## 10. Análisis de residuos

Además del gráfico de valores reales frente a predichos, se revisan los residuos del modelo seleccionado.

Un residuo representa la diferencia entre el valor real y el valor estimado. Este gráfico permite observar si los errores presentan algún patrón evidente.

In [ ]:
residuos = y_test - pred_mejor

plt.figure(figsize=(8, 5))
plt.scatter(pred_mejor, residuos, alpha=0.6)
plt.axhline(0, linestyle="--")
plt.title(f"Residuos del modelo - {mejor_nombre}")
plt.xlabel("Hectáreas predichas")
plt.ylabel("Residuo (hectáreas)")
plt.show()

## 11. Revisión de predicciones negativas

Debido a que el número de hectáreas afectadas no puede ser negativo, se revisa si alguno de los modelos genera estimaciones menores que cero.

La presencia de estos valores representa una limitación del modelo y debe considerarse al interpretar los resultados.

In [ ]:
negativos = int((pred_mejor < 0).sum())

print("Predicciones negativas:", negativos)

if negativos > 0:
    print(
        "El modelo genera algunas predicciones negativas. "
        "Estas estimaciones no tienen interpretación física válida y representan una limitación."
    )
else:
    print("El modelo seleccionado no generó predicciones negativas en el conjunto de prueba.")

## 12. Conclusión

La ampliación del dataset permite entrenar y comparar más de un algoritmo de regresión utilizando información de varios años.

La comparación con un baseline ayuda a determinar si los modelos aportan una mejora real. Los resultados deben interpretarse como una aproximación académica a la magnitud de incendios ya registrados.